# osu!mania Difficulty Model - Colab Training

Use this when local training is too slow. In VS Code, click the kernel picker and choose `Colab` -> `New Colab Server`, then pick a GPU runtime and run top to bottom.


In [ ]:
# Setup
REPO_URL = "https://github.com/fatelvx/ODP-model.git"
BRANCH = ""  # Optional. Leave empty to use the repo default branch.

import os
import subprocess
from pathlib import Path

if not REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repo URL first.")

!rm -rf /content/mania-difficulty-model
clone_cmd = ["git", "clone", "--depth", "1"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [REPO_URL, "/content/mania-difficulty-model"]
subprocess.run(clone_cmd, check=True)
%cd /content/mania-difficulty-model
!pip install -e .


In [ ]:
# Check GPU
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
!nvidia-smi


## Smoke Test

Run this first. It proves the training/report pipeline works before spending API calls or GPU time on real data.


In [ ]:
!python -m mania_difficulty.tools.make_synthetic_dataset --maps 128 --out data/processed/synthetic_colab
!python -m mania_difficulty.train \
  --labels data/processed/synthetic_colab/labels.csv \
  --sequences data/processed/synthetic_colab/sequences \
  --epochs 12 \
  --batch-size 32 \
  --run-name colab_synthetic \
  --model lstm \
  --device cuda


In [ ]:
from IPython.display import Image, display, HTML

run_dir = Path("outputs/runs/colab_synthetic")
display(Image(str(run_dir / "learning_curve.png")))
display(Image(str(run_dir / "prediction_scatter.png")))
display(HTML((run_dir / "run_report.html").read_text(encoding="utf-8")))


## Real osu! Data

Use this after the smoke test. The credentials live only in this Colab session.


In [ ]:
import getpass
os.environ["OSU_CLIENT_ID"] = getpass.getpass("OSU_CLIENT_ID: ")
os.environ["OSU_CLIENT_SECRET"] = getpass.getpass("OSU_CLIENT_SECRET: ")


In [ ]:
# Start smaller than 2000 while debugging. Raise --target after the pipeline is stable.
!python -m mania_difficulty.data.fetch_maps --target 300 --out data/raw/beatmaps.csv
!python -m mania_difficulty.data.fetch_osu_files --maps data/raw/beatmaps.csv --out-dir data/raw/osu
!python -m mania_difficulty.data.parse_notes --maps data/raw/beatmaps.csv --osu-dir data/raw/osu --out-dir data/processed/sequences
!python -m mania_difficulty.data.fetch_scores --maps data/raw/beatmaps.csv --out data/processed/labels.csv --min-scores 30


In [ ]:
import json
from pathlib import Path

!python -m mania_difficulty.tools.sweep_forest \
  --labels data/processed/labels.csv \
  --sequences data/processed/sequences \
  --out-dir outputs/forest_sweep_top100 \
  --cv-folds 5 \
  --group-column beatmapset_id \
  --trees 200,500 \
  --min-samples-leaf 1,2,4 \
  --max-features sqrt,0.75 \
  --feature-sets core,burst \
  --workers -1

best_params = json.loads(Path("outputs/forest_sweep_top100/best_params.json").read_text(encoding="utf-8"))
FOREST_TREES = int(best_params["forest_trees"])
FOREST_MIN_SAMPLES_LEAF = int(best_params["forest_min_samples_leaf"])
FOREST_MAX_FEATURES = str(best_params["forest_max_features"])
FEATURE_SET = str(best_params.get("feature_set", "core"))
best_params


In [ ]:
!python -m mania_difficulty.train \
  --labels data/processed/labels.csv \
  --sequences data/processed/sequences \
  --run-name colab_forest_top100 \
  --model tabular_forest \
  --forest-trees {FOREST_TREES} \
  --forest-min-samples-leaf {FOREST_MIN_SAMPLES_LEAF} \
  --forest-max-features {FOREST_MAX_FEATURES} \
  --feature-set {FEATURE_SET} \
  --cv-folds 5 \
  --group-column beatmapset_id \
  --workers -1


In [ ]:
!python -m mania_difficulty.train \
  --labels data/processed/labels.csv \
  --sequences data/processed/sequences \
  --epochs 50 \
  --batch-size 32 \
  --run-name colab_lstm_top100 \
  --model lstm \
  --group-column beatmapset_id \
  --device cuda


In [ ]:
from google.colab import files
!python -m mania_difficulty.tools.compare_runs outputs/runs/colab_forest_top100 outputs/runs/colab_lstm_top100 --out-csv outputs/colab_top100_comparison.csv --out-html outputs/colab_top100_comparison.html
!zip -r colab_top100_outputs.zip outputs/forest_sweep_top100 outputs/runs/colab_forest_top100 outputs/runs/colab_lstm_top100 outputs/colab_top100_comparison.csv outputs/colab_top100_comparison.html
files.download("colab_top100_outputs.zip")


## Inspect Real Runs

Use this to compare metrics, human-review candidates, plots, and HTML reports after training.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import FileLink, HTML, Image, display

sweep_dir = Path("outputs/forest_sweep_top100")
if sweep_dir.exists():
    print("Forest sweep best params")
    display(json.loads((sweep_dir / "best_params.json").read_text(encoding="utf-8")))
    display(pd.read_csv(sweep_dir / "sweep_summary.csv"))
    display(HTML((sweep_dir / "sweep_report.html").read_text(encoding="utf-8")))

run_names = ["colab_forest_top100", "colab_lstm_top100"]
metric_rows = []
run_rows = []
for run_name in run_names:
    run_dir = Path("outputs/runs") / run_name
    for metrics_name, evaluation in [("metrics.json", "holdout"), ("cv_metrics.json", "cv_oof")]:
        metrics_path = run_dir / metrics_name
        if not metrics_path.exists():
            continue
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        for target, values in metrics.items():
            if isinstance(values, dict) and "mae" in values:
                metric_rows.append({"run": run_name, "evaluation": evaluation, "target": target, **values})
        run_rows.append({"run": run_name, "evaluation": evaluation, **metrics["_run"]})

metrics_frame = pd.DataFrame(metric_rows).sort_values(["evaluation", "target", "mae"])
display(metrics_frame)
display(pd.DataFrame(run_rows))

for run_name in run_names:
    run_dir = Path("outputs/runs") / run_name
    print("\nRUN", run_name)
    history = pd.read_csv(run_dir / "history.csv")
    review_path = run_dir / "cv_human_review.csv"
    if not review_path.exists():
        review_path = run_dir / "human_review.csv"
    human_review = pd.read_csv(review_path)
    print("Last 10 training epochs")
    display(history.tail(10))
    print("Rows to human-review first", review_path.name)
    display(human_review.head(20))
    display(Image(str(run_dir / "prediction_scatter.png")))
    cv_scatter = run_dir / "cv_prediction_scatter.png"
    if cv_scatter.exists():
        display(Image(str(cv_scatter)))
    display(Image(str(run_dir / "learning_curve.png")))
    feature_plot = run_dir / "feature_importance.png"
    if feature_plot.exists():
        display(Image(str(feature_plot)))
    display(HTML((run_dir / "run_report.html").read_text(encoding="utf-8")))

display(FileLink("colab_top100_outputs.zip"))
